Unlike ESG, where several standardised lexicons exist, no academically consolidated lexicon currently captures DEI or AI disclosure in corporate filings. We therefore construct bespoke lexicons grounded in recurring disclosure language observed in 10-K filings, regulatory guidance, and prior empirical studies on corporate signalling.

These terms capture disclosure regimes that are not already in E/S/G, and any residual overlap is explicitly controlled for. They are sized and sub-categorised according to the same structure (100 words / 10 categories) as the standard ESG lexicon. As AI/DEI pillars are addressed separately there is no noise caused by varying total lexicon length.

In [10]:
import yaml
import polars as pl
import plotly.express as px

# Load data
df = pl.read_parquet("spy_10k_2015_present.parquet")

# Load lexicon (regex already defined in YAML)
with open("AI_DEI_Lexicon.yml", "r", encoding="utf-8") as f:
    lex = yaml.safe_load(f)

ai_patterns  = lex["ai"]
dei_patterns = lex["dei"]

ticker = "AAPL"
section = "risk_factors"

plot_df = (
    df
    .filter((pl.col("ticker") == ticker) & (pl.col("section") == section))
    .sort("filing_date")
    .with_columns(
        # Sum regex hits across all AI patterns
        pl.sum_horizontal(
            [pl.col("text").str.count_matches(p) for p in ai_patterns]
        ).alias("AI"),

        # Sum regex hits across all DEI patterns
        pl.sum_horizontal(
            [pl.col("text").str.count_matches(p) for p in dei_patterns]
        ).alias("DEI"),
    )
    .select(["filing_date", "AI", "DEI"])
    .to_pandas()
)

fig = px.line(
    plot_df,
    x="filing_date",
    y=["AI", "DEI"],
    title=f"AI vs DEI Lexicon Occurrences — {ticker} ({section})",
    labels={"value": "Count", "filing_date": "Filing Date"},
    markers=True,
)

fig.update_layout(template="plotly_white")
fig.update_yaxes(tickmode="linear", tick0=0, dtick=1)
fig.show()


In [16]:
import re
import yaml
import polars as pl
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "vscode"

TICKER = "AAPL"
START_YEAR = 2015
END_YEAR = 2024

LEXICON_FILE = "AI_DEI_Lexicon.yml"
PARQUET_FILE = "spy_10k_2015_present.parquet"

with open(LEXICON_FILE, "r", encoding="utf-8") as f:
    lex = yaml.safe_load(f)

AI_PATTERNS = lex["ai"]
DEI_PATTERNS = lex["dei"]

df = pl.read_parquet(PARQUET_FILE, columns=["ticker", "filing_period", "text"])

df = (
    df.with_columns(
        pl.col("filing_period").cast(pl.Date),
        pl.col("filing_period").dt.year().alias("year"),
    )
    .filter(
        (pl.col("ticker") == TICKER)
        & (pl.col("year") >= START_YEAR)
        & (pl.col("year") <= END_YEAR)
        & pl.col("text").is_not_null()
    )
)

# What this does: turn common ESG-style regex into readable labels
def regex_to_label(p: str) -> str:
    s = p

    # Strip inline flag and common anchors
    s = re.sub(r"^\(\?i\)", "", s)
    s = s.replace(r"\b", "")
    s = s.strip()

    # Turn common “space/hyphen” patterns into spaces
    s = s.replace(r"[-\s]+", " ").replace(r"[-\s]", " ")

    # Remove escaping that makes it unreadable
    s = s.replace(r"\.", ".").replace(r"\-", "-").replace(r"\s", " ")

    # Remove basic grouping syntax for display
    s = re.sub(r"\(\?:", "(", s)          # non-capturing -> capturing (display)
    s = re.sub(r"\(\s*\)", "", s)         # empty groups
    s = re.sub(r"[()]", "", s)            # drop parentheses
    s = s.replace("?:", "")

    # Clean quantifiers that are just plural markers
    s = s.replace("s?", "s")
    s = re.sub(r"\{\d+,?\d*\}", "", s)    # drop {m,n}

    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()

    # Small cosmetics
    s = s.replace("a.i.", "A.I.")
    if s.lower() == "ai":
        return "AI"
    if s.lower() == "dei":
        return "DEI"
    if s.lower() == "llm s" or s.lower() == "llms":
        return "LLMs"

    return s

def plot_bucket(bucket_name: str, patterns: list[str]):
    cols = [f"t_{i:03d}" for i in range(len(patterns))]

    tmp = df.with_columns(
        [pl.col("text").str.count_matches(patterns[i]).alias(cols[i]) for i in range(len(patterns))]
    )

    wide = (
        tmp.group_by("year")
          .agg([pl.sum(c).alias(c) for c in cols])
          .sort("year")
    )

    # What this does: compute total hits per term over all years
    totals = []
    for i, c in enumerate(cols):
        total = wide.select(pl.col(c).sum()).item()
        total = 0 if total is None else int(total)
        if total > 0:
            totals.append((i, c, total))

    if not totals:
        print(f"{bucket_name}: no non-zero terms for {TICKER} in {START_YEAR}-{END_YEAR}")
        return

    # What this does: choose 6–8 labels (top by total hits)
    totals.sort(key=lambda x: x[2], reverse=True)
    k = len(totals)
    top_n = min(8, k)
    if top_n < 6:
        top_n = k  # if fewer than 6 non-zero terms exist, just show what's there

    keep = totals[:top_n]

    # What this does: build readable labels; ensure uniqueness
    used = set()
    rename_map = {}
    for i, c, _ in keep:
        base = regex_to_label(patterns[i])
        label = base
        j = 2
        while label.lower() in used:
            label = f"{base} ({j})"
            j += 1
        used.add(label.lower())
        rename_map[c] = label

    plot_df = (
        wide.select(["year"] + [c for _, c, _ in keep])
            .rename(rename_map)
            .to_pandas()
            .melt(id_vars="year", var_name="term", value_name="count")
    )

    fig = px.line(
        plot_df,
        x="year",
        y="count",
        color="term",
        markers=True,
        title=f"{bucket_name} Term Occurrences in {TICKER} 10-K (Entire Report, {START_YEAR}–{END_YEAR})",
        labels={"year": "Filing Year", "count": "Occurrences", "term": f"{bucket_name} Term"},
    )
    fig.update_layout(template="plotly_white", hovermode="x unified")
    fig.update_xaxes(tickmode="linear", tick0=START_YEAR, dtick=1)
    fig.update_yaxes(tickmode="linear", tick0=0)
    fig.show()

plot_bucket("AI", AI_PATTERNS)
plot_bucket("DEI", DEI_PATTERNS)


AAPL historically talks product experience, design, and platform integration far more than it talks “AI” explicitly. Even when they are using ML heavily, they frame it as features (camera quality, on-device intelligence, privacy-preserving UX) rather than as “AI strategy”. So only 3–4 AI terms surviving across the full report is totally consistent with how Apple writes.

In practical terms, in 2015–2019 corporate filings generally didn’t use explicit DEI terminology as often as later years, even if the company had internal DEI policies.
Before 2020:

Human capital disclosure was largely prescriptive and minimal

Firms disclosed headcount, nothing more

Workforce composition, diversity, equity, inclusion → implicitly non-material

After August 2020:

The SEC replaced prescriptive rules with a principles-based requirement

Firms must disclose human capital measures that are material to their business

Crucially: the definition of “material” was left to firms and investors